# Check the live services

Optional: check weather, currency conversion, itinerary planning and image generation. This sends real requests and saves results in `integration_smoke`.

In [ ]:
from advisor import TravelAdvisor, external_tool


In [ ]:
"""Explicit paid integration smoke test. Never run automatically from the notebook."""

import json
import os
from pathlib import Path
from dotenv import load_dotenv

### Run the checks

Change `run_live_checks` to `True` to make the paid API calls.

In [ ]:
run_live_checks = False

if run_live_checks:
    load_dotenv()
    key = os.environ.get('OPENAI_API_KEY', '').strip()
    if not key:
        raise ValueError('Set OPENAI_API_KEY first.')
    output = Path('integration_smoke')
    output.mkdir(exist_ok=True)
    checks = {}
    for name, arguments in [('get_weather_forecast', {'city': 'Lisbon, Portugal'}),
                             ('get_exchange_rate', {'amount': 200, 'from_currency': 'EUR', 'to_currency': 'JPY'})]:
        try:
            data = external_tool(name, arguments)
            checks[name] = {'success': True, 'source': data['source'], 'source_url': data['source_url']}
        except (RuntimeError, ValueError, KeyError):
            checks[name] = {'success': False}
        print(f"{name}: {checks[name]['success']}", flush=True)
    advisor = TravelAdvisor(key, budget_usd=.20)
    try:
        reply = advisor.reply('Plan 2 days in Lisbon, Portugal. I enjoy architecture and local food. Moderate budget. Include a currency conversion of EUR 200 to JPY using the current reference rate.')
        (output / 'itinerary.md').write_text(reply, encoding='utf-8')
        checks['itinerary'] = {'success': bool(advisor.last_itinerary), 'models_used': list(dict.fromkeys(m['model'] for m in advisor.metrics))}
        print(f"Itinerary handoff: {checks['itinerary']}", flush=True)
        if advisor.last_itinerary:
            try:
                (output / 'destination.png').write_bytes(advisor.generate_image())
                checks['image'] = {'success': True}
            except (RuntimeError, ValueError) as error:
                checks['image'] = {'success': False, 'error': str(error)}
            print(f"Image: {checks.get('image')}", flush=True)
    except (RuntimeError, ValueError) as error:
        checks['itinerary'] = {'success': False, 'error': str(error)}
        print(f"Itinerary: {checks['itinerary']}", flush=True)
    checks['cost_usd'] = advisor.cost_usd
    checks['metrics'] = advisor.metrics
    (output / 'results.json').write_text(json.dumps(checks, indent=2), encoding='utf-8')